# 04. 台湾の深掘り：なぜ残差がこれほど大きいのか

生産性指数とPPP残差の関係について、水準（レベル）の相関だけでなく
前年差・対数差分でも確認する（両者ともトレンドを持つ時系列であり、水準相関だけでは
見せかけの相関になりうるため）。また、03で追加した年固定効果付き回帰の残差を使う。

03で、台湾は高所得クラブ40か国中、残差が最小（-51.9%、年固定効果付きでは
-50.6%）という結果になった。この章では、台湾統計局の産業別生産性データを使い、
2つの角度から掘り下げる。

1. 台湾の**工業（貿易財に近い）の生産性**は、経済全体の所得の伸びよりどれだけ速いか。
2. 古典的なBalassa-Samuelson効果は「貿易財の生産性↑→物価水準↑」を予測する。
   台湾でこの関係が実際に観察されるかを、水準・変化率の両方で確認する。

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
sys.path.insert(0, str(BASE_DIR))

from src.taiwan_data import annualize_productivity_index, load_taiwan_industry_productivity

TAIWAN_FILE = RAW_DIR / "taiwan_labor_productivity_raw.xlsx"
if not TAIWAN_FILE.exists():
    print(f"台湾統計局データが見つかりません: {(TAIWAN_FILE).relative_to(BASE_DIR)}")

## 1. 台湾統計局データの整形（民国暦→西暦、月次→年次）

サービス業（服務業）は物量ベースの生産性指数が定義できず全期間欠測のため、工業のみを扱う。

In [2]:
monthly = load_taiwan_industry_productivity(TAIWAN_FILE)
annual = annualize_productivity_index(monthly)
print(f"年次データ（6か月以上そろう年のみ）: {len(annual)}年分、{annual['year'].min()}〜{annual['year'].max()}年")

年次データ（6か月以上そろう年のみ）: 45年分、1982〜2026年


## 2. 台湾の工業生産性 vs 経済全体の所得の伸び（水準）

In [3]:
pwt = pd.read_csv(PROCESSED_DIR / "pwt_relative_price_income.csv")
twn_pwt = pwt[pwt["countrycode"] == "TWN"].sort_values("year")

merged = annual.merge(twn_pwt[["year", "rgdpe_pc", "relp", "reli"]], on="year", how="inner")
print(f"台湾統計局×PWTが重なる年: {merged['year'].min()}〜{merged['year'].max()}年（{len(merged)}年分）")

base_year = merged["year"].min()
base_row = merged[merged["year"] == base_year].iloc[0]
merged["industry_productivity_index_100"] = (
    merged["industry_productivity_index_annual"] / base_row["industry_productivity_index_annual"] * 100
)
merged["gdp_per_capita_index_100"] = merged["rgdpe_pc"] / base_row["rgdpe_pc"] * 100

n_years = merged["year"].max() - base_year
industry_cagr = (merged["industry_productivity_index_100"].iloc[-1] / 100) ** (1 / n_years) - 1
gdp_cagr = (merged["gdp_per_capita_index_100"].iloc[-1] / 100) ** (1 / n_years) - 1

print(f"\n{base_year}年を100とした場合の{merged['year'].max()}年時点の指数:")
print(f"  工業の労働生産性: {merged['industry_productivity_index_100'].iloc[-1]:.1f}（年率{industry_cagr*100:+.2f}%）")
print(f"  一人当たりGDP（購買力平価）: {merged['gdp_per_capita_index_100'].iloc[-1]:.1f}（年率{gdp_cagr*100:+.2f}%）")

台湾統計局×PWTが重なる年: 1982〜2023年（42年分）

1982年を100とした場合の2023年時点の指数:
  工業の労働生産性: 602.5（年率+4.48%）
  一人当たりGDP（購買力平価）: 476.0（年率+3.88%）


## 3. 生産性の伸びとPPP残差の関係：水準・前年差・対数差分

残差は03の**年固定効果付き**プール回帰のものを使う。水準の相関に加えて、
両系列の前年差（レベルのトレンドを除いた変化）でも相関を確認する。
前年差の相関がゼロに近ければ、水準の相関は主に共通トレンド（見せかけの相関）に
由来する可能性が高い。

In [4]:
pool_fe_fit = pd.read_csv(PROCESSED_DIR / "penn_regression_pooled_year_fe.csv")
twn_resid = pool_fe_fit[pool_fe_fit["countrycode"] == "TWN"][["year", "resid_pct"]].sort_values("year")

compare = merged.merge(twn_resid, on="year", how="inner").sort_values("year").reset_index(drop=True)
print("台湾：年別の工業生産性指数（基準年=100） と PPP残差（年固定効果付き、負ほど回帰予測値を下回る）")
print(compare[["year", "industry_productivity_index_100", "resid_pct"]].round(1).to_string(index=False))

# (a) 水準の相関
level_corr = compare[["industry_productivity_index_100", "resid_pct"]].corr().iloc[0, 1]

# (b) 前年差の相関
compare["d_productivity"] = compare["industry_productivity_index_100"].diff()
compare["d_resid_pct"] = compare["resid_pct"].diff()
diff_corr = compare[["d_productivity", "d_resid_pct"]].corr().iloc[0, 1]

# (c) 対数差分（成長率）の相関
compare["log_productivity"] = np.log(compare["industry_productivity_index_100"])
compare["dlog_productivity"] = compare["log_productivity"].diff()
log_diff_corr = compare[["dlog_productivity", "d_resid_pct"]].corr().iloc[0, 1]

print(f"\n水準の相関係数: {level_corr:.3f}")
print(f"前年差の相関係数: {diff_corr:.3f}（n={compare['d_productivity'].notna().sum()}）")
print(f"対数差分×残差前年差の相関係数: {log_diff_corr:.3f}")

if abs(diff_corr) < 0.3:
    print(
        "\n→ 前年差の相関は水準の相関より大幅に弱い。水準相関は主に両系列に共通する"
        "トレンド（見せかけの相関）を反映している可能性が高く、年々の変化同士の"
        "関係としては明確な関係を確認できない。"
    )
else:
    print("\n→ 前年差でも一定の相関が残る。")

台湾：年別の工業生産性指数（基準年=100） と PPP残差（年固定効果付き、負ほど回帰予測値を下回る）
 year  industry_productivity_index_100  resid_pct
 2005                            359.1      -54.0
 2006                            372.4      -54.2
 2007                            396.4      -57.7
 2008                            394.1      -54.6
 2009                            398.6      -55.6
 2010                            463.3      -56.6
 2011                            468.5      -54.9
 2012                            468.3      -54.0
 2013                            483.0      -55.1
 2014                            503.4      -56.2
 2015                            500.5      -51.1
 2016                            530.8      -48.8
 2017                            546.6      -45.5
 2018                            555.7      -45.3
 2019                            559.2      -43.2
 2020                            610.0      -46.9
 2021                            687.7      -46.6
 2022                            678.5      -45

## 4. 為替管理という別の説明（定性的な背景）

§3の結果を踏まえ、生産性の水準トレンドと残差の水準トレンドが同時に観察されるという
事実を、為替管理という定性的な背景と併せて提示する（相関・因果の検証ではない）。

- 台湾の外貨準備は2025年9月末の6,029.4億ドルから、11月末には5,997.9億ドルへ
  減少した（台湾中央銀行、外貨準備高発表資料）。台湾中央銀行は変動要因として、
  (1) 外貨準備運用収益、(2) 米ドル以外の準備通貨の対ドル為替変動、(3) 外国為替
  市場の秩序維持のための介入、の3点を挙げているが、介入の方向・寄与額は
  公表していない（同発表資料）。この外貨準備の増減は2025年9〜11月の事例であり、
  本節が扱う2023年時点の残差を同時期に説明する証拠ではない、後年の参考事例
  である点に注意する。
- 2025年11月、『エコノミスト』誌の「ビッグマック指数で55%割安」という指摘に対し、
  台湾中央銀行は反論し、米財務省との間で「為替操作をしない」共同声明を発表した。

## 5. 保存

In [5]:
out_path = PROCESSED_DIR / "taiwan_productivity_vs_ppp_residual.csv"
compare.to_csv(out_path, index=False)
print(f"保存先: {(out_path).relative_to(BASE_DIR)}")

保存先: data\processed\taiwan_productivity_vs_ppp_residual.csv


## まとめと限界

- 台湾の工業生産性は、経済全体の所得の伸びを上回るペースで伸びている
  （1982〜2023年、年率+4.48% vs +3.88%）。
- 生産性指数の水準とPPP残差の水準には正の相関（+0.766）が見られるが、**前年差での相関は
  大幅に弱く**、水準相関の少なくとも一部は両系列に共通するトレンド（見せかけの
  相関）に由来すると考えられる。したがって「古典的B-S効果では説明できない」と
  断定することは避け、**「工業生産性指数の上昇とPPP残差の低下が、同じ期間に
  水準として併存した」**という記述にとどめる。
- 為替管理・輸出構造の集中といった定性的な背景は、相関や因果を検証したものではなく、
  残差の解釈を助ける文脈情報として扱う。
- **データの限界**：工業の生産性指数は物量ベースであり、非貿易財（サービス業）との
  比較ができない。ここでの『生産性の伸び』は、経済全体の所得の伸びとの比較に
  留まる（真の意味での貿易財/非貿易財の生産性格差ではない）。
- **時系列の自己相関**：前年差の系列自体にも自己相関が残っている可能性があり、
  本稿ではNewey-West等による正式な検定は行っていない。

次は `05_japan_taiwan_residual.ipynb` で、日本・台湾の残差比較をまとめ、
エコノミスト誌の論争との対応を整理する。